## Crew allocation and EPANET controls
This notebook is meant to test variables and parameters when generating Crew allocation and EPANET controls.
Crew allocation to the pipes (indexes) is generated in random order.
It also imports the functions from crews_and_controls_BPDRR.py 

In [4]:
from pprint import pprint
import random
import pandas as pd
import numpy as np

import crews_and_controls_BPDRR

In [5]:
# Damage scenario selection
ds_sel = 'DS5'

df = pd.read_excel(
    "DS_with_full_description.xlsx",
    sheet_name= ds_sel
)

# Repair time discretization (hours)
repair_time_interval = 0.25  # 15 minutes

# Build the reparations dictionary
time_reparation = {}

for _, row in df.iterrows():

    repair_time = float(row["fix time (hours)"])

    # Round up to the nearest interval
    repair_time = np.ceil(repair_time / repair_time_interval) * repair_time_interval

    time_reparation[str(row["Pipe ID"])] = {
        "t_r": repair_time
    }

print(f"{len(time_reparation)} repairs loaded.")
#print(list(time_reparation.items())[5:10])
time_reparation

108 repairs loaded.


{'3922': {'t_r': np.float64(7.25)},
 '3398': {'t_r': np.float64(7.25)},
 '3825': {'t_r': np.float64(7.25)},
 '4117': {'t_r': np.float64(7.25)},
 '1902': {'t_r': np.float64(5.75)},
 '3019': {'t_r': np.float64(5.75)},
 '91': {'t_r': np.float64(5.75)},
 '1540': {'t_r': np.float64(4.5)},
 '2050': {'t_r': np.float64(4.5)},
 '2115': {'t_r': np.float64(4.5)},
 '378': {'t_r': np.float64(4.5)},
 '4129': {'t_r': np.float64(4.5)},
 '1028': {'t_r': np.float64(3.5)},
 '4594': {'t_r': np.float64(3.5)},
 '2162': {'t_r': np.float64(3.5)},
 '5105': {'t_r': np.float64(3.5)},
 '1402': {'t_r': np.float64(3.5)},
 '4652': {'t_r': np.float64(3.5)},
 '1282': {'t_r': np.float64(3.5)},
 '1631': {'t_r': np.float64(3.5)},
 '2404': {'t_r': np.float64(3.5)},
 '246': {'t_r': np.float64(3.5)},
 '2804': {'t_r': np.float64(3.5)},
 '3136': {'t_r': np.float64(3.5)},
 '3807': {'t_r': np.float64(3.5)},
 '4748': {'t_r': np.float64(3.5)},
 '475': {'t_r': np.float64(3.5)},
 '5004': {'t_r': np.float64(3.5)},
 '5458': {'t_r': n

In [6]:
# load the wdn as INP file WITH the broken pipes
input_inp = 'BBM-EPS_'+ds_sel+'mcg.inp'

# Export the new INP file with the controls
output_inp="BBM-EPS_"+ds_sel+"_restoration.inp"

In [7]:
# Number of crews
n_teams = 3

# Pipe IDs from the reparations dictionary
pipe_ids = list(time_reparation.keys())

# Number of repairs
n_rep = len(pipe_ids)

In [10]:
# Import the travel time matrix from the Excel file
dmatrix_df = pd.read_parquet(
    "TravelTime_"+ds_sel+".parquet",
#    sheet_name= ds_sel,
#    index_col=0
)

# Normalize IDs so the repair dictionary and matrix use the same keys
dmatrix_df.index = dmatrix_df.index.map(str)
dmatrix_df.columns = dmatrix_df.columns.map(str)

dmatrix_df.head()

,3922,3398,3825,4117,1902,3019,91,1540,2050,2115,...,5280,5432,5523,5550,5845,587,5893,772,90,98
3922,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,...,0.75,0.75,0.75,0.75,0.75,0.25,0.75,0.25,0.25,0.25
3398,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,...,0.75,0.75,0.50,0.50,0.50,0.25,0.50,0.25,0.25,0.25
3825,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,...,0.75,0.75,0.75,0.50,0.50,0.25,0.50,0.25,0.25,0.25
4117,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.25,...,0.75,0.75,0.75,0.75,0.75,0.25,0.75,0.25,0.50,0.25
1902,0.25,0.25,0.25,0.25,0.25,0.25,0.25,0.50,0.25,0.25,...,0.75,0.75,0.75,0.50,0.75,0.25,0.75,0.25,0.25,0.25


In [11]:
indexes = random.sample(pipe_ids, n_rep)
print('Show permutation of reparations')
print(indexes)
print(len(indexes))

Show permutation of reparations
['5004', '4129', '2212', '3398', '3136', '2651', '3121', '809', '253', '5845', '3825', '2626', '4065', '98', '1640', '5022', '5280', '1418', '5207', '4805', '587', '4316', '5893', '2596', '3720', '4889', '1857', '3049', '2646', '1028', '335', '1352', '3615', '378', '3922', '3807', '6012', '2972', '4652', '90', '1060', '91', '1032', '1926', '1902', '5100', '3536', '1402', '2115', '838', '3216', '772', '4594', '4117', '390', '3842', '1936', '88', '3723', '1989', '3762', '3613', '721', '1794', '1631', '5550', '3409', '1540', '1947', '5194', '4093', '5458', '5688', '3019', '2068', '475', '5865', '4379', '2600', '4022', '1007', '2772', '3690', '3754', '6005', '3800', '3547', '1731', '4671', '2804', '1832', '1558', '3123', '5105', '246', '153', '626', '2464', '2404', '2162', '5432', '60', '5523', '4748', '2050', '1282', '4386', '154']
108


In [12]:
# apply the greedy allocation to the dataset
crews, new_controls = crews_and_controls_BPDRR.allocate_crews(time_reparation, dmatrix_df, indexes, n_teams = n_teams)
#print('')
#print('Show the allocation of reparations to crews')
#pprint(crews)
print('')
print('[CONTROLS]')
pprint(new_controls)


organize distances for each crew
Crew 3, from 2212-to-3398 : 0.25
Crew 1, from 5004-to-3136 : 0.5
Crew 2, from 4129-to-2651 : 0.25
Crew 1, from 3136-to-3121 : 0.25
Crew 2, from 2651-to-809 : 0.25
Crew 3, from 3398-to-253 : 0.25
Crew 1, from 3121-to-5845 : 0.75
Crew 2, from 809-to-3825 : 0.25
Crew 3, from 253-to-2626 : 0.25
Crew 1, from 5845-to-4065 : 0.75
Crew 3, from 2626-to-98 : 0.25
Crew 1, from 4065-to-1640 : 0.25
Crew 2, from 3825-to-5022 : 0.5
Crew 3, from 98-to-5280 : 0.5
Crew 1, from 1640-to-1418 : 0.5
Crew 3, from 5280-to-5207 : 0.25
Crew 2, from 5022-to-4805 : 0.25
Crew 1, from 1418-to-587 : 0.25
Crew 3, from 5207-to-4316 : 0.75
Crew 1, from 587-to-5893 : 0.5
Crew 3, from 4316-to-2596 : 0.25
Crew 1, from 5893-to-3720 : 0.5
Crew 2, from 4805-to-4889 : 0.25
Crew 3, from 2596-to-1857 : 0.25
Crew 1, from 3720-to-3049 : 0.25
Crew 2, from 4889-to-2646 : 0.75
Crew 3, from 1857-to-1028 : 0.5
Crew 1, from 3049-to-335 : 0.25
Crew 2, from 2646-to-1352 : 0.5
Crew 3, from 1028-to-3615 : 

In [13]:
# Apply function to generate INP with controls
controls_inp = crews_and_controls_BPDRR.write_inp_controls(
    input_inp=input_inp,
    output_inp=output_inp,
    crews=crews,
    new_controls=new_controls
)

print(f"Created: {controls_inp}")

Restoration INP successfully created.
Output file : BBM-EPS_DS5_restoration.inp
Controls    : 432
Created: BBM-EPS_DS5_restoration.inp
